# Extração de Dados SRAG

Download e verificação do Parquet canônico `dados-v2/srag.parquet`.

In [1]:
from pathlib import Path
import sys

import pandas as pd


ROOT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from src import config as env
from src import config as C
from src import dados as storage
from src.dados import get_parquet


env.carregar_dotenv()
env.garantir_diretorios()

print("ROOT_DIR:", ROOT_DIR)
print(env.resumo_ambiente())

ROOT_DIR: c:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
ROOT_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
ENV               = local
.env encontrado?  = False
GITHUB_TOKEN?     = False


In [2]:
print(f"Repositório:  {C.GITHUB_REPO}")
print(f"Versão:       {C.DATA_VERSION}")
print(f"Arquivo:      {C.PARQUET_NAME}")
print(f"URL canônica: {C.srag_parquet_url()}")

Repositório:  Ivan-S-Oliveira/Projeto_Integrador
Versão:       dados-v2
Arquivo:      srag.parquet
URL canônica: https://github.com/Ivan-S-Oliveira/Projeto_Integrador/releases/download/dados-v2/srag.parquet


In [3]:
parquet_path = get_parquet()

print(f"\nParquet em: {parquet_path}")
print(f"Tamanho:    {parquet_path.stat().st_size / 1024**2:.2f} MB")

Baixando: 304/304 MB
[storage] SHA-256: 96ed6b9d5ceebaaea00659f77511112174b04bb980cb243b560c00dd2a8f316c

Parquet em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed\dados-v2\srag.parquet
Tamanho:    289.51 MB


In [4]:
import time
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

inicio=time.perf_counter()
arquivo=pq.ParquetFile(parquet_path)
meta=arquivo.metadata
quantidade_registros=int(meta.num_rows)
quantidade_colunas=int(meta.num_columns)
quantidade_grupos_linhas=int(meta.num_row_groups)
tamanho_arquivo_mb=parquet_path.stat().st_size/1_048_576

if quantidade_registros<=0 or quantidade_colunas<=0:
    raise ValueError("Parquet vazio ou sem colunas.")

lote=next(arquivo.iter_batches(batch_size=5),None)
if lote is None:
    raise ValueError("Não foi possível extrair a amostra.")
amostra_parquet=lote.to_pandas(types_mapper=pd.ArrowDtype).reset_index(drop=True)
memoria_amostra_mb=amostra_parquet.memory_usage(index=True,deep=True).sum()/1_048_576

resumo_ingestao=pd.DataFrame({"metrica":["caminho","tamanho_mb","registros","colunas","row_groups","linhas_amostra","memoria_amostra_mb","tempo_s"],
"valor":[str(parquet_path),round(tamanho_arquivo_mb,2),quantidade_registros,quantidade_colunas,quantidade_grupos_linhas,len(amostra_parquet),round(memoria_amostra_mb,4),round(time.perf_counter()-inicio,3)]})
display(amostra_parquet)
display(resumo_ingestao)
print("Verificação concluída sem carregar o dataset completo.")

,id_registro,ano_arquivo,data_snapshot,arquivo_origem,nu_notific,dt_notific,sem_not,dt_sin_pri,sem_pri,sg_uf_not,...,vg_oms,vg_omsout,vg_lin,vg_met,vg_metout,vg_dtres,vg_enc,vg_reinf,vg_codest,reinf
0,0,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31781201987679,2026-06-11,24,2026-06-02,23,BA,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
1,1,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31775652598724,2026-03-23,13,2026-03-22,13,MA,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
2,2,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31781205405890,2026-03-29,14,2026-03-23,13,MT,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
3,3,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31782407823707,2026-06-17,25,2026-06-15,25,SP,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2
4,4,2026,2026-09-28,INFLUD26-28-09-2026.parquet,31773493207022,2026-03-14,11,2026-03-09,11,SE,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2


,metrica,valor
0,caminho,C:\Users\ivanI\OneDrive\Documentos\Projetos\Pr...
1,tamanho_mb,289.51
2,registros,4666910
3,colunas,198
4,row_groups,28
5,linhas_amostra,5
6,memoria_amostra_mb,0.006
7,tempo_s,0.091


Verificação concluída sem carregar o dataset completo.
